# Patent Reference — the citation edge list (`citing_id`, `cited_id`, `type`, `grant_id`)

The **edges themselves**, not counts of them. `patent_citation.ipynb` and
`patent_citation_trend.ipynb` both scan the same two files and immediately collapse them —
into per-target window counts, and into per-target/per-year counts. This notebook stops one
step earlier and writes the reference network out, one row per citation, so anything that
needs the graph (co-citation, bibliographic coupling, disruption, path analysis) can read it
instead of re-scanning 250M rows of TSV.

## Raw data
```
/project/jevans/Dawoon/Science of Science/PatentView/Granted
├── g_patent.tsv.zip                    # patent_id, patent_type              -> the utility universe
├── g_us_patent_citation.tsv.zip        # patent_id(citing), citation_patent_id(cited)
└── g_us_application_citation.tsv.zip   # patent_id(citing), citation_document_number(cited pgpub)
/project/jevans/Dawoon/Science of Science/PatentView/Pregranted
└── pg_granted_pgpubs_crosswalk.tsv.zip # pgpub_id -> patent_id
```

## Output
`/project/jevans/Dawoon/Science of Science/PatentView/output/patent_reference.parquet`

| column | meaning |
|---|---|
| `citing_id` | granted `patent_id` making the reference |
| `cited_id`  | **the identifier as it was actually cited** — a granted `patent_id` when `type='granted'`, a **pre-grant publication number** (`pgpub_id`, 11 digits) when `type='application'` |
| `type`      | `granted` — the reference appears on the granted patent (`g_us_patent_citation`)<br>`application` — it appears in the citing patent's application-stage record (`g_us_application_citation`), where the cited entity is the pgpub |
| `grant_id`  | the **granted `patent_id` the cited document is / became**. Equal to `cited_id` on `granted` rows; resolved through `pg_granted_pgpubs_crosswalk` on `application` rows. `NULL` only when `KEEP_UNMAPPED_PGPUB` is on and the cited application never issued |

`cited_id` preserves what the citing document pointed at; `grant_id` is the *entity* it points
to. That split is the whole reason this column exists: the same reference made at application
stage and repeated at grant carries two different identifiers (`20030139363` and `7462862`),
so before, collapsing both onto the granted id made the two rows look like the same edge seen
twice. Now the pgpub is visible and `grant_id` is what joins them.

**Join on `grant_id`, not `cited_id`** — it is the granted-patent endpoint, so it is what
lines up with `patent_metadata.parquet`, `patent_citation.parquet` and every other output
here. `citing_id` is always a granted `patent_id` and needs no such care.

```python
ref = pd.read_parquet(OUT/'patent_reference.parquet')
ref[ref.type == 'granted']                                   # the classic citation network
ref.groupby('grant_id').size()                               # every reference ever made, per patent
ref.drop_duplicates(['citing_id', 'grant_id'])               # the union, each edge once
```

## What is and is not filtered

Three conventions are switches at the top of the next cell, because a raw edge list should
say plainly what it dropped:

- `UTILITY_ONLY` — both endpoints must be a US **utility** patent listed in `g_patent`.
  On the application side the test is applied to `grant_id`, not to the pgpub in `cited_id`.
  Drops design/plant/reissue patents.
- `EXCLUDE_THIRD_PARTY` — drops `cited by third party` rows (AIA pre-issuance submissions),
  as `patent_citation` and `patent_citation_trend` do, so in-degrees computed here reproduce
  those files.
- `KEEP_UNMAPPED_PGPUB` — **off**. An application-stage citation to a pgpub that never issued
  has no granted patent to point at. It used to be dropped because `cited_id` had nowhere to
  put it; now that `cited_id` holds the pgpub itself, such a row *can* be represented, as
  `grant_id IS NULL`. It is still dropped by default, so row counts stay comparable with
  `patent_citation.parquet` and `UTILITY_ONLY` keeps its meaning on both endpoints. Turn it
  on to get the ~25M references to applications that never became patents — with the
  understanding that nothing about the cited side of those rows can be checked or joined.

Deliberately **not** filtered, unlike the counting notebooks:

- **No `diff >= 0` filter.** Those notebooks keep only citations where the citing patent's
  grant year is at or after the cited patent's, because a negative lag makes no sense in a
  forward-citation *window*. As an edge it is still a real reference (the citing application
  was published before the cited patent issued), so it is kept here. Filter on grant years
  downstream if a window is what you want.
- **No provenance split.** `citation_category` is used only for the third-party exclusion;
  examiner / non-examiner / unknown lives in `patent_citation.parquet`.

Rows are de-duplicated on `(citing_id, cited_id, type)` — `grant_id` is a function of the
first two and adds nothing to the key. A reference that appears in *both* the application
record and the granted patent — the normal case, since a citing patent usually repeats its
references at grant — appears **twice**: once as `(pgpub, application)` and once as
`(patent_id, granted)`, sharing a `grant_id`. That is the point of the two columns:
`type='granted'` alone is the classic patent citation network, the union is every reference
ever made, and the intersection — `GROUP BY citing_id, grant_id HAVING count(DISTINCT type)=2`
— is what `uniqueC` in `patent_citation.ipynb` is correcting for.

One consequence of keying on the pgpub: a citing patent that cites **two different pgpubs of
the same eventual patent** (a continuation republished, say) now produces two rows where it
used to produce one. Those are genuinely distinct references, they are counted in the checks
below, and `drop_duplicates(['citing_id','grant_id'])` collapses them if you want edges
rather than references.

## Memory — this does not run on a login node

`g_us_patent_citation` is **152.6M** rows and `g_us_application_citation` **78.6M**. Everything
below streams in chunks and never holds the edge list in memory — parquet parts go to
`cache/reference_edges/`, and duckdb does the de-duplication and the final sort out of core.

Even so, the midway3 login nodes put an **8 GiB cgroup limit on the entire user session**, and
the resident lookups alone (the 8.5M-entry utility index plus the 5.4M-entry pgpub crosswalk)
are ~2.7 GB before a single chunk is parsed. A login-node run is OOM-killed. Submit it:

```
cd '/project/jevans/Dawoon/Science of Science/PatentView'
sbatch run_patent_reference.sbatch          # -p jevans, 200G, ~15 min
```

which runs these same cells through `../run_notebook.py` with `PV_CHUNK` and `PV_DUCKDB_MEM`
raised. Measured throughput: ~3s per 1M granted rows, ~11s per 1M application rows.

In [1]:
import os, sys, gc, time, glob, shutil
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv

ROOT, D, OUT = pv.BASE, pv.GRANTED, pv.OUT
OUT_FP = pv.out('patent_reference.parquet')
XWALK  = pv.pregranted('pg_granted_pgpubs_crosswalk.tsv.zip')

UTILITY_ONLY        = True          # both endpoints must be a utility patent in g_patent
EXCLUDE_THIRD_PARTY = True          # drop 'cited by third party' (AIA pre-issuance submissions)
KEEP_UNMAPPED_PGPUB = False         # keep application citations to pgpubs that never issued,
                                    # as grant_id NULL. Off: they are dropped, as before, so
                                    # counts stay comparable with patent_citation.parquet.
# Sized for wherever this is running. The midway3 login nodes cap the whole user session
# (this kernel AND everything else you have open) at 8 GiB, which is not enough to finish --
# submit run_patent_reference.sbatch instead, which raises both through the environment.
CHUNK      = int(os.environ.get('PV_CHUNK', 2_000_000))    # rows per read_csv chunk
DUCKDB_MEM = os.environ.get('PV_DUCKDB_MEM', '4GB')        # duckdb's own budget

# Parquet parts, one per input chunk. Written instead of accumulated because the union is
# ~2.5e8 rows of strings; the final de-duplicated file is assembled from these by duckdb.
PART_DIR = f'{pv.CACHE}/reference_edges'
os.makedirs(PART_DIR, exist_ok=True)
for _f in glob.glob(f'{PART_DIR}/*.parquet'):
    os.remove(_f)                   # a stale half-written log would silently corrupt the union

def write_part(citing, cited, grant, kind, i):
    '''Append one chunk of edges. `type` is dictionary-encoded: it is one repeated value.
    `grant` is the granted-patent endpoint -- the same array as `cited` for granted rows,
    the crosswalked patent_id (or None) for application rows.'''
    if not len(citing):
        return 0
    pq.write_table(
        pa.table({'citing_id': pa.array(citing, pa.string()),
                  'cited_id':  pa.array(cited,  pa.string()),
                  'type': pa.DictionaryArray.from_arrays(
                      pa.array(np.zeros(len(citing), np.int32)), pa.array([kind])),
                  'grant_id': pa.array(grant, pa.string())}),
        f'{PART_DIR}/{kind}_{i:05d}.parquet', compression='zstd')
    return len(citing)

pv.preflight('patent_reference')

granted    : /project/jevans/Dawoon/Science of Science/PatentView/Granted   (35 zip files)
pregranted : /project/jevans/Dawoon/Science of Science/PatentView/Pregranted   (25 files)
output     : /project/jevans/Dawoon/Science of Science/PatentView/output

  patent_reference                OK


## 1. The utility-patent universe

`pd.Index` keeps its hash table alive after the first lookup, so `get_indexer` over each
chunk is a hash probe rather than a rebuild — the membership test runs ~30 times per file.

In [2]:
%%time
gp = pd.read_csv(os.path.join(D, 'g_patent.tsv.zip'), sep='\t',
                 usecols=['patent_id', 'patent_type'], dtype=str)
util = pd.Index(gp.loc[gp['patent_type'] == 'utility', 'patent_id'].dropna().unique())
util.get_indexer(util[:1])                      # force the hash table once, up front
print(f'utility patents: {len(util):,}  (of {len(gp):,} granted patents)')
del gp; gc.collect()

def in_universe(*cols):
    '''Boolean mask: every column's id is a known utility patent.'''
    m = np.ones(len(cols[0]), bool)
    for c in cols:
        m &= util.get_indexer(c) >= 0
    return m

utility patents: 8,531,961  (of 9,454,161 granted patents)


## 2. Granted references — `g_us_patent_citation`

`patent_id` cites `citation_patent_id`; both sides are already granted patent ids, so no
crosswalk is needed and `grant_id` is just `cited_id` again. This is the classic patent
citation network.

In [3]:
%%time
seen = kept = drop_na = drop_tp = drop_univ = 0
t0 = time.time()
for i, ch in enumerate(pd.read_csv(os.path.join(D, 'g_us_patent_citation.tsv.zip'), sep='\t',
                                   usecols=['patent_id', 'citation_patent_id', 'citation_category'],
                                   dtype=str, chunksize=CHUNK)):
    seen += len(ch)
    n = len(ch); ch = ch.dropna(subset=['patent_id', 'citation_patent_id']); drop_na += n - len(ch)
    if EXCLUDE_THIRD_PARTY:
        keep = ~ch['citation_category'].str.contains('third party', case=False, na=False, regex=False)
        drop_tp += int((~keep).sum()); ch = ch.loc[keep]
    citing = ch['patent_id'].values; cited = ch['citation_patent_id'].values
    if UTILITY_ONLY:
        m = in_universe(citing, cited)
        drop_univ += int((~m).sum()); citing, cited = citing[m], cited[m]
    kept += write_part(citing, cited, cited, 'granted', i)   # cited IS the granted id here
    del ch, citing, cited

print(f'[{time.time()-t0:.0f}s] g_us_patent_citation: {seen:,} rows read')
print(f'    dropped  missing id       {drop_na:>12,}')
print(f'    dropped  third party      {drop_tp:>12,}')
print(f'    dropped  non-utility end  {drop_univ:>12,}')
print(f'    KEPT                      {kept:>12,}   ({kept/max(seen,1)*100:.1f}%)')
n_granted_rows = kept

[309s] g_us_patent_citation: 152,631,929 rows read
    dropped  missing id                 21
    dropped  third party             5,426
    dropped  non-utility end    33,333,265
    KEPT                       119,293,217   (78.2%)


## 3. Application references — `g_us_application_citation`

References made at the **application (pre-grant) stage**. The citing side is a granted
`patent_id`; the cited side is a pre-grant publication number
(`citation_document_number` = `pgpub_id`), which is kept **as `cited_id`** and separately
resolved through `pg_granted_pgpubs_crosswalk` into the patent it eventually issued as,
**`grant_id`**.

A cited pgpub whose application never issued has no `grant_id`. That is ~32% of the rows read
here — the same ~68% mapping rate `patent_citation.ipynb` reports — and by default it is
still dropped, because `UTILITY_ONLY` has nothing to test and the counts would stop agreeing
with `patent_citation.parquet`. Set `KEEP_UNMAPPED_PGPUB = True` above to keep them with a
null `grant_id` instead.

In [4]:
%%time
xw = pd.read_csv(XWALK, sep='\t', usecols=['pgpub_id', 'patent_id'], dtype=str, on_bad_lines='skip')
xw = xw.dropna(subset=['pgpub_id', 'patent_id']).drop_duplicates('pgpub_id')
pgpub2pid = dict(zip(xw['pgpub_id'], xw['patent_id']))
print(f'crosswalk pgpub->patent pairs: {len(pgpub2pid):,}')
del xw; gc.collect()

seen = kept = drop_na = drop_tp = drop_xw = drop_univ = n_null_grant = 0
t0 = time.time()
for i, ch in enumerate(pd.read_csv(os.path.join(D, 'g_us_application_citation.tsv.zip'), sep='\t',
                                   usecols=['patent_id', 'citation_document_number', 'citation_category'],
                                   dtype=str, chunksize=CHUNK, on_bad_lines='skip')):
    seen += len(ch)
    n = len(ch); ch = ch.dropna(subset=['patent_id', 'citation_document_number']); drop_na += n - len(ch)
    if EXCLUDE_THIRD_PARTY:
        keep = ~ch['citation_category'].str.contains('third party', case=False, na=False, regex=False)
        drop_tp += int((~keep).sum()); ch = ch.loc[keep]
    citing = ch['patent_id'].values
    cited  = ch['citation_document_number'].values                   # the pgpub, kept as cited_id
    grant  = ch['citation_document_number'].map(pgpub2pid).values    # -> granted id, NaN if never issued
    mapped = pd.notna(grant)
    if KEEP_UNMAPPED_PGPUB:
        n_null_grant += int((~mapped).sum())
        grant = np.where(mapped, grant, None)        # NaN -> None, so parquet stores a real null
    else:
        drop_xw += int((~mapped).sum())
        citing, cited, grant = citing[mapped], cited[mapped], grant[mapped]
    if UTILITY_ONLY:
        # The utility test belongs to the granted endpoint; an unmapped pgpub has none to test,
        # so it is exempted rather than silently failing the lookup. With KEEP_UNMAPPED_PGPUB
        # off there are no nulls left and this is exactly the old `in_universe(citing, cited)`.
        gna = pd.isna(grant)
        m = (util.get_indexer(citing) >= 0) & ((util.get_indexer(np.where(gna, '', grant)) >= 0) | gna)
        drop_univ += int((~m).sum()); citing, cited, grant = citing[m], cited[m], grant[m]
    kept += write_part(citing, cited, grant, 'application', i)
    del ch, citing, cited, grant

del pgpub2pid; gc.collect()
print(f'[{time.time()-t0:.0f}s] g_us_application_citation: {seen:,} rows read')
print(f'    dropped  missing id            {drop_na:>12,}')
print(f'    dropped  third party           {drop_tp:>12,}')
if KEEP_UNMAPPED_PGPUB:
    print(f'    KEPT     cited pgpub unmapped  {n_null_grant:>12,}   (grant_id NULL)')
else:
    print(f'    dropped  cited pgpub unmapped  {drop_xw:>12,}')
print(f'    dropped  non-utility end       {drop_univ:>12,}')
print(f'    KEPT                           {kept:>12,}   ({kept/max(seen,1)*100:.1f}%)')
n_app_rows = kept

parts = glob.glob(f'{PART_DIR}/*.parquet')
print(f'\nedge parts: {len(parts)} files, '
      f'{sum(os.path.getsize(p) for p in parts)/1e9:.2f} GB in {PART_DIR}')

crosswalk pgpub->patent pairs: 5,432,615
[317s] g_us_application_citation: 78,562,544 rows read
    dropped  missing id                       0
    dropped  third party                  4,824
    dropped  cited pgpub unmapped    24,795,995
    dropped  non-utility end          1,329,607
    KEPT                             52,432,118   (66.7%)

edge parts: 23 files, 1.62 GB in /project/jevans/Dawoon/Science of Science/PatentView/cache/reference_edges


## 4. De-duplicate and write

`SELECT DISTINCT` over the union, out of core. `grant_id` is functionally determined by
`(cited_id, type)` — it is `cited_id` itself on granted rows and a crosswalk lookup, deduped
on `pgpub_id`, on application rows — so adding it to the projection does not change the key.
Duplicates within a source are rare but real (the same reference listed twice on one patent);
duplicates *across* sources are common and are **kept**, because they differ in `type` and in
`cited_id` while sharing a `grant_id` — that is what the two columns are for.

Sorted by `grant_id` first so in-degree scans and predicate pushdown on a cited patent touch
a handful of row groups instead of the whole file, and so both the granted row and the
application row for one reference land next to each other.

In [5]:
%%time
import duckdb
con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEM}'")
con.execute(f"SET temp_directory='{pv.CACHE}/duckdb_tmp'")
con.execute("SET preserve_insertion_order=false")     # lets the sort and the write stream

con.execute(f'''
COPY (
  SELECT DISTINCT citing_id, cited_id, CAST(type AS VARCHAR) AS type, grant_id
  FROM read_parquet('{PART_DIR}/*.parquet')
  ORDER BY grant_id, cited_id, citing_id, type
) TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)
''')
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e9:.2f} GB)')

summary = con.execute(f'''
SELECT type, count(*) AS refs,
       count(DISTINCT citing_id) AS citing_patents,
       count(DISTINCT cited_id)  AS cited_docs,
       count(DISTINCT grant_id)  AS cited_patents,
       count(*) FILTER (WHERE grant_id IS NULL) AS null_grant_id
FROM read_parquet('{OUT_FP}') GROUP BY type ORDER BY type
''').fetchdf()
total = con.execute(f"SELECT count(*) FROM read_parquet('{OUT_FP}')").fetchone()[0]
print(f'\nrows: {total:,}   (scanned {n_granted_rows + n_app_rows:,} before de-duplication, '
      f'{(n_granted_rows + n_app_rows - total):,} exact duplicates removed)')
print('  cited_docs counts pgpubs for application rows and patent_ids for granted rows;\n'
      '  cited_patents counts the granted patents behind them, which is the comparable number.')
display(summary)

WROTE /project/jevans/Dawoon/Science of Science/PatentView/output/patent_reference.parquet  (0.78 GB)

rows: 171,532,249   (scanned 171,725,335 before de-duplication, 193,086 exact duplicates removed)
  cited_docs counts pgpubs for application rows and patent_ids for granted rows;
  cited_patents counts the granted patents behind them, which is the comparable number.


,type,refs,citing_patents,cited_docs,cited_patents,null_grant_id
0,application,52432036,4593182,3896626,3892693,0
1,granted,119100213,7438443,5973369,5973369,0


## 5. Checks

Four things worth knowing before anyone builds on this file: that the two sources overlap the
way they should, how often the pgpub key splits one edge into two references, that in-degrees
here reproduce `patent_citation.parquet`, and what the top of the distribution looks like.

Every one of these groups on `grant_id`, not `cited_id` — which is the habit downstream code
should copy.

In [6]:
%%time
# 1. overlap -- references recorded in both the application record and the granted patent.
#    Keyed on grant_id, because the two rows carry different cited_ids by construction.
ov = con.execute(f'''
SELECT count(*) FROM (
  SELECT citing_id, grant_id FROM read_parquet('{OUT_FP}') WHERE grant_id IS NOT NULL
  GROUP BY citing_id, grant_id HAVING count(DISTINCT type) = 2)
''').fetchone()[0]
uniq_pairs = con.execute(f'''SELECT count(DISTINCT (citing_id, grant_id))
                             FROM read_parquet('{OUT_FP}') WHERE grant_id IS NOT NULL''').fetchone()[0]
print(f'distinct (citing, grant) edges      {uniq_pairs:>12,}')
print(f'  recorded in both sources          {ov:>12,}   ({ov/uniq_pairs*100:.1f}% of edges)')
print(f'  -> summing granted + application rows double counts these; that is what '
      f'uniqueC in patent_citation.ipynb corrects.')

# 2. the cost of keying cited_id on the pgpub: one citing patent citing two pgpubs of the same
#    eventual patent is two references here and was one row before the grant_id column existed.
multi = con.execute(f'''
SELECT count(*) AS edges, sum(n) - count(*) AS extra_rows FROM (
  SELECT citing_id, grant_id, count(DISTINCT cited_id) AS n
  FROM read_parquet('{OUT_FP}') WHERE type = 'application' AND grant_id IS NOT NULL
  GROUP BY citing_id, grant_id HAVING count(DISTINCT cited_id) > 1)
''').fetchone()
print(f'\n(citing, grant) edges cited via >1 pgpub  {multi[0] or 0:>12,}'
      f'   (+{multi[1] or 0:,} rows vs. keying on the granted id)')

# 3. agreement with patent_citation.parquet. Its C_all / appC_all keep only citations with a
#    non-negative grant-year lag, which this file deliberately does not filter -- so the
#    comparison is made on the same footing by re-applying the lag filter here. Counted as
#    DISTINCT (citing, grant) so the pgpub split in check 2 does not show up as a difference.
CIT_FP = pv.out('patent_citation.parquet')
if os.path.exists(CIT_FP):
    gy = pd.read_csv(os.path.join(D, 'g_patent.tsv.zip'), sep='\t',
                     usecols=['patent_id', 'patent_type', 'patent_date'], dtype=str)
    gy = gy[gy['patent_type'] == 'utility']
    gy['grant_year'] = pd.to_datetime(gy['patent_date'], errors='coerce').dt.year
    gy = gy.dropna(subset=['grant_year'])[['patent_id', 'grant_year']]
    gy['grant_year'] = gy['grant_year'].astype('int32')
    con.register('gy', gy)
    deg = con.execute(f'''
    SELECT e.type, count(DISTINCT (e.citing_id, e.grant_id)) AS edges
    FROM read_parquet('{OUT_FP}') e
    JOIN gy a ON a.patent_id = e.citing_id
    JOIN gy b ON b.patent_id = e.grant_id
    WHERE a.grant_year >= b.grant_year
    GROUP BY e.type ORDER BY e.type
    ''').fetchdf()
    cit = pd.read_parquet(CIT_FP, columns=['C_all', 'appC_all']).sum()
    here = dict(zip(deg['type'], deg['edges']))
    print(f'\n{"":<14}{"patent_reference":>18}{"patent_citation":>18}{"delta":>14}')
    for k, ref in (('granted', 'C_all'), ('application', 'appC_all')):
        h, c = here.get(k, 0), int(cit[ref])
        print(f'{k:<14}{h:>18,}{c:>18,}{h-c:>14,}   ({ref})')
    print('  (patent_citation counts citation ROWS, this file counts DISTINCT edges, so the\n'
          '   delta should be <= 0 and equal to the duplicate rows patent_citation counts twice.)')
    del gy; gc.collect()
else:
    print(f'\n{CIT_FP} not built yet -- skipping the cross-check.')

# 4. shape of the network
display(con.execute(f'''
SELECT grant_id, count(*) FILTER (WHERE type='granted')     AS granted_refs,
                 count(*) FILTER (WHERE type='application') AS application_refs
FROM read_parquet('{OUT_FP}') WHERE grant_id IS NOT NULL GROUP BY grant_id
ORDER BY granted_refs + application_refs DESC LIMIT 10
''').fetchdf())
# both rows of one reference, side by side: same grant_id, different cited_id and type
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') LIMIT 10").fetchdf())

distinct (citing, grant) edges       164,881,051
  recorded in both sources             6,629,728   (4.0% of edges)
  -> summing granted + application rows double counts these; that is what uniqueC in patent_citation.ipynb corrects.

(citing, grant) edges cited via >1 pgpub        21,205   (+21,470 rows vs. keying on the granted id)

                patent_reference   patent_citation         delta
granted              119,098,182       119,291,183      -193,001   (C_all)
application           49,520,554        49,541,000       -20,446   (appC_all)
  (patent_citation counts citation ROWS, this file counts DISTINCT edges, so the
   delta should be <= 0 and equal to the duplicate rows patent_citation counts twice.)


,grant_id,granted_refs,application_refs
0,7462862,3939,3901
1,7453065,3917,3918
2,7468304,3928,3885
3,7049190,3919,3886
4,4683202,5807,0
5,7674650,4317,1160
6,4683195,5259,0
7,7663607,3285,1956
8,7601984,1021,4005
9,5523520,4910,0


,citing_id,cited_id,type,grant_id
0,10388323,20150201176,application,10244223
1,10416452,20150201176,application,10244223
2,10432944,20150201176,application,10244223
3,10440354,20150201176,application,10244223
4,10448030,20150201176,application,10244223
5,10453431,20150201176,application,10244223
6,10469833,20150201176,application,10244223
7,10477189,20150201176,application,10244223
8,10491887,20150201176,application,10244223
9,10495859,20150201176,application,10244223


## 6. Drop the intermediate parts

The parts under `cache/reference_edges/` are the pre-de-duplication log and are reproducible
by re-running cells 2–3. Removed once the output is on disk, since they are several GB.

In [7]:
con.close()
if os.path.exists(OUT_FP) and os.path.getsize(OUT_FP) > 0:
    freed = sum(os.path.getsize(p) for p in glob.glob(f'{PART_DIR}/*.parquet'))
    shutil.rmtree(PART_DIR)
    print(f'removed {PART_DIR}  ({freed/1e9:.2f} GB freed)')
    print(f'kept    {OUT_FP}  ({os.path.getsize(OUT_FP)/1e9:.2f} GB)')
else:
    print('output missing or empty -- parts kept')

removed /project/jevans/Dawoon/Science of Science/PatentView/cache/reference_edges  (1.62 GB freed)
kept    /project/jevans/Dawoon/Science of Science/PatentView/output/patent_reference.parquet  (0.78 GB)
